In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-11-07 22:47:41,821 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-11-07 22:47:41,821 INFO sqlalchemy.engine.Engine COMMIT


✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

The original 4 articles were expanded to **16** in `data/external/cultpass_articles.jsonl` (reservations & 24h cancellation policy, blocked accounts, refunds/billing, premium tiers, quota, payment methods, app/QR troubleshooting, profile/e-mail changes, guests, venue cancellations, accessibility, privacy/LGPD).

In [7]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [8]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [9]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [10]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [11]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [12]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [13]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


**Past ticket + long-term memory** for Bob Stone (`f556c0`), so the agents can recall a previous interaction and a stated preference in a brand-new conversation.

In [14]:
from datetime import datetime, timedelta

past_ticket_id = "past-ticket-bob-001"
with get_session(engine) as session:
    bob = udahub.User(
        user_id=str(uuid.uuid4()),
        account_id=account_id,
        external_user_id=cultpass_users[1]["id"],
        user_name=cultpass_users[1]["name"],
    )
    past = datetime.now() - timedelta(days=20)
    session.add_all([
        bob,
        udahub.Ticket(ticket_id=past_ticket_id, account_id=account_id, user_id=bob.user_id,
                      channel="email", created_at=past),
        udahub.TicketMetadata(ticket_id=past_ticket_id, status="resolved",
                              main_issue_type="experience_info", tags="premium, experiences"),
        udahub.TicketMessage(message_id=str(uuid.uuid4()), ticket_id=past_ticket_id, role="user",
                             content="Are premium experiences included in my plan? I love the samba events in Rio.",
                             created_at=past),
        udahub.TicketMessage(message_id=str(uuid.uuid4()), ticket_id=past_ticket_id, role="ai",
                             content="Yes! As a Premium member, premium experiences are included in your monthly quota.",
                             created_at=past + timedelta(minutes=1)),
        udahub.CustomerMemory(memory_id=str(uuid.uuid4()), account_id=account_id, user_id=bob.user_id,
                              ticket_id=past_ticket_id, memory_type="interaction", created_at=past,
                              content=f"{past.date()} [experience_info] Asked whether premium experiences are included -> resolved: included in Premium tier.",
                              tags="premium"),
        udahub.CustomerMemory(memory_id=str(uuid.uuid4()), account_id=account_id, user_id=bob.user_id,
                              ticket_id=past_ticket_id, memory_type="preference", created_at=past,
                              content="Enjoys samba / music events in Rio de Janeiro", tags="preference"),
    ])

# Tests

In [15]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='23663ac2-ddb0-4071-bad6-9037b5ec867c', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='19c1d236-169c-4df2-b1ef-37190b180734', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='07ace0fc-d733-4866-a117-47ca63d2610c', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='8a030b3d-028d-44d0-b4d2-98efcc187195', title='How to Handle Login Issues?')>
<Knowledge(article_id='c7df9189-fc66-45e7-9510-5af0c1fc4a4a', title='How to Cancel a Reservation (24-Hour Policy)')>
<Knowledge(article_id='f9403196-f7fe-4057-ab20-400554d4788e', title='Account Blocked or Suspended')>
<Knowledge(article_id='d08ff3f1-95c4-40fd-9516-4285a7a34035', title='Refund Requests and Billing Disputes')>
<Knowledge(article_id='37c1b76d-54de-4870-8647-1b52d9388ca1', title='Premium Experiences and Subscription Tiers')>
<Knowledge(article_id='36b247c3-04a2-443f-947b-03222391ca3a', title='Monthly Quota and Unused Experiences')>
<Knowledge(articl

In [17]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='d35886a7-dbf5-4b49-a343-b6c97d796914', user_name='Alice Kingsley', external_user_id='a4ab87')>
<User(user_id='7912e849-c9ae-469a-a43e-de8963d2104b', user_name='Bob Stone', external_user_id='f556c0')>


In [18]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='8a1030f1-a8d5-4ea5-b7e5-33aa9e534a57', role='user', content='I can't log in to my Cultpass ...')>


In [19]:
with get_session(engine) as session:
    print("articles:", session.query(udahub.Knowledge).count())
    for memory in session.query(udahub.CustomerMemory).all():
        print(memory)

articles: 16
<CustomerMemory(memory_id='bd105e00-0f54-422e-86e0-f8591e28d6db', type='interaction', content='2026-10-18 [experience_info] A')>
<CustomerMemory(memory_id='a09a3ba3-9eb4-4dba-a28e-0db9c339b345', type='preference', content='Enjoys samba / music events in')>


**Vector store (semantic search)**: article embeddings are stored in `knowledge_embeddings` and only recomputed when an article changes. Built here when an API key is configured; otherwise retrieval falls back to TF-IDF.

In [20]:
from dotenv import load_dotenv
from agentic import config
from agentic.tools import udahub_tools

load_dotenv()
result = udahub_tools.search_knowledge("my QR code does not show", k=2)
print("retrieval mode:", result["mode"], "| threshold:", result["min_relevance"])
for hit in result["results"]:
    print(f"  {hit['score']:.3f}  {hit['title']}")
with get_session(engine) as session:
    print("stored vectors:", session.query(udahub.KnowledgeEmbedding).count())

retrieval mode: hybrid | threshold: 0.2
  0.519  App Troubleshooting: Crashes and QR Code Not Showing
  0.226  How to Reserve a Spot for an Event
stored vectors: 16
